---
last_verified: 2026-10-03
tool_version: n/a
---

# Choosing between on-premise and cloud modes in GitGuardian

> Deployment-mode reference. Release-neutral by design: no product version is
> claimed here, and nothing below depends on behaviour specific to one release.
> Names used for scanner components are illustrative, not schema claims.

## Purpose

Secret scanning has to run somewhere. Either a hosted service receives the
scan content and returns findings, or a self-managed installation keeps both
the content and the findings inside infrastructure the team controls. The
detection logic is comparable in both shapes; what differs is where code
travels, who operates the scanning service, and which failure modes the team
owns. This notebook compares the two shapes on data movement, operational
cost, change management, and diagnostic difficulty, then supplies a scoring
function for choosing between them for a given estate.

## When to use

Reach for this comparison when deciding:

- whether scan content may leave the team's network or must stay inside it;
- whether the team can absorb running another stateful service (upgrades,
  backups, availability) or would rather consume scanning as a service;
- how a scanning outage or a bad configuration change is detected and rolled
  back in each shape;
- which shape fits each repository tier, since the answer is rarely the same
  for every tier.

## Prerequisites

- A rough inventory of what gets scanned: repository tiers, approximate
  commit volume, and any data-residency constraint per tier.
- Knowledge of who is on call for the scanning path in each candidate shape.
- Nothing needs to be installed to run this notebook: every code cell is a
  self-contained model with no network or service dependencies.


## Steps

### 1. Model the two deployment shapes

The comparison starts from what actually moves. In the hosted shape, the
scanner ships findings-relevant content to the vendor service and the
dashboard, alerting, and remediation tracking live there. In the
self-managed shape, content stays inside the team's network and the team
operates the service that stores findings and serves the dashboard. The cell
below encodes both shapes as data so the later scoring cells can reason
about them explicitly.


In [ ]:
# last_verified: 2026-10-03 · gitguardian n/a

# Deployment-shape model. Field names are illustrative, not a schema claim.

CLOUD = {
    "name": "hosted-service",
    "content_leaves_network": True,
    "team_operates_stateful_service": False,
    "dashboard_owner": "vendor",
    "upgrade_owner": "vendor",
    "availability_owner": "vendor",
    "per_repo_onboarding": "register repository in the hosted console",
}

SELF_MANAGED = {
    "name": "self-managed-installation",
    "content_leaves_network": False,
    "team_operates_stateful_service": True,
    "dashboard_owner": "team",
    "upgrade_owner": "team",
    "availability_owner": "team",
    "per_repo_onboarding": "point the scanner at the internal service",
}

SHAPES = (CLOUD, SELF_MANAGED)

for shape in SHAPES:
    print(f"{shape['name']}: leaves_network={shape['content_leaves_network']}, "
          f"team_operates_service={shape['team_operates_stateful_service']}")

# The shapes differ on every ownership axis; there is no third shape in
# this comparison, only a per-tier choice between these two.
assert CLOUD["content_leaves_network"] is True
assert SELF_MANAGED["content_leaves_network"] is False
assert CLOUD["team_operates_stateful_service"] is False
assert SELF_MANAGED["team_operates_stateful_service"] is True


### 2. Score the data-movement constraint

Data movement is usually the deciding axis, and it is also the least
negotiable one. A tier whose content must not leave the network eliminates
the hosted shape outright, regardless of how attractive its operational cost
looks. A tier with no such constraint keeps both shapes on the table. The
cell encodes that gate: residency is a veto, not a weight.


In [ ]:
# Residency gate: a must-stay-inside tier vetoes the hosted shape.
# Tiers are illustrative labels for the scoring model.

TIERS = [
    {"tier": "illustrative-public-samples", "must_stay_inside": False},
    {"tier": "illustrative-internal-services", "must_stay_inside": False},
    {"tier": "illustrative-regulated-workloads", "must_stay_inside": True},
]


def allowed_shapes(tier, shapes=SHAPES):
    """Return the shapes a tier may use given its residency constraint."""
    if tier["must_stay_inside"]:
        return [s for s in shapes if not s["content_leaves_network"]]
    return list(shapes)


for tier in TIERS:
    allowed = [s["name"] for s in allowed_shapes(tier)]
    print(f"{tier['tier']}: {allowed}")

assert [s["name"] for s in allowed_shapes(TIERS[0])] == [
    "hosted-service",
    "self-managed-installation",
]
assert [s["name"] for s in allowed_shapes(TIERS[2])] == [
    "self-managed-installation"
]


### 3. Score the operational cost

Where residency does not decide, operations usually do. The hosted shape
trades a per-seat or per-volume service cost for near-zero operational load:
no upgrades to schedule, no finding store to back up, no scanner fleet to
keep healthy. The self-managed shape inverts the trade: the licence or
service cost drops away, but the team owns upgrades, storage, availability,
and the on-call path when scanning stops producing findings. The cell models
that trade as recurring burden items per shape so the choice stays explicit
instead of vibes-based.


In [ ]:
# Operational-burden model: recurring items each shape imposes on the team.
# Weights are relative effort markers for comparison, not measured values.

BURDEN = {
    "hosted-service": {
        "schedule upgrades": 0,
        "back up finding store": 0,
        "keep scanner fleet healthy": 1,  # runners/agents still run locally
        "manage vendor relationship and access": 2,
        "prove residency compliance per audit": 2,
    },
    "self-managed-installation": {
        "schedule upgrades": 3,
        "back up finding store": 2,
        "keep scanner fleet healthy": 3,
        "manage vendor relationship and access": 0,
        "prove residency compliance per audit": 1,  # content never left
    },
}


def total_burden(shape_name):
    return sum(BURDEN[shape_name].values())


for name in BURDEN:
    print(f"{name}: total burden {total_burden(name)}")
    for item, weight in BURDEN[name].items():
        print(f"  - {item}: {weight}")

# Self-managed always costs the team more operationally; if it did not,
# there would be no trade-off to model.
assert total_burden("self-managed-installation") > total_burden("hosted-service")
# The hosted shape never asks the team to upgrade or back up the service.
assert BURDEN["hosted-service"]["schedule upgrades"] == 0
assert BURDEN["hosted-service"]["back up finding store"] == 0


### 4. Change management and outage behaviour

| Change | Hosted service | Self-managed installation |
|---|---|---|
| Roll out a stricter scanning policy | Edit once in the hosted console; applies on next scan | Edit and redeploy the internal service, then confirm every scanner picked it up |
| Scope of a bad policy change | Every connected repository over-blocks until reverted in one place | Only estates pointing at that service instance; a staged rollout can limit blast radius |
| Rollback | Revert the policy object in the console | Redeploy the previous service revision and policy bundle |
| Scanning silently stops | Vendor status page plus a local dead-man check on finding freshness | Team-owned monitoring must notice; no outside party will |
| Diagnostic surface | Vendor-supplied run logs and delivery receipts | Team-owned logs end to end, including the service internals |

The asymmetry to plan for: the hosted shape centralises both control and
failure, while the self-managed shape distributes control and concentrates
the on-call burden. Either way, a freshness check on findings ("did every
expected repository produce scan results in the last interval") belongs in
monitoring, because a scanner that reports nothing looks identical to a
scanner that found nothing.


### 5. Score a concrete estate

Rather than arguing the choice in the abstract, score it per tier. The
function below takes the properties a tier actually has and returns the
recommended shape plus the reasoning that produced it. Residency is a veto
applied first; only surviving shapes are scored on operations. It is
deliberately simple: a weighted score is only useful if the weights are
visible and arguable.


In [ ]:
def choose_shape(tier):
    """Return (recommendation, reasons) for a repository tier.

    Recognised tier keys:
      must_stay_inside   content must not leave the team's network (veto)
      scan_volume         rough scans/day across the tier
      team_runs_services  team already operates stateful services (bool)
      audit_pressure      residency/compliance audits are frequent (bool)
    """
    reasons = []
    candidates = allowed_shapes(tier)
    names = [s["name"] for s in candidates]

    if tier.get("must_stay_inside"):
        reasons.append("residency vetoes the hosted shape for this tier")
        return "self-managed-installation", reasons

    hosted = 0
    managed = 0

    if tier.get("audit_pressure"):
        managed += 2
        reasons.append("frequent audits favour content that never left")
    if tier.get("team_runs_services"):
        managed += 2
        reasons.append("team already operates stateful services, so the "
                         "self-managed burden is marginal")
    else:
        hosted += 2
        reasons.append("no service-operation practice in the team; hosted "
                         "shape avoids creating an on-call burden")

    volume = tier.get("scan_volume", 0)
    if volume >= 500:
        managed += 1
        reasons.append(f"~{volume} scans/day makes per-volume service cost "
                         "worth revisiting against self-managed burden")
    else:
        hosted += 1
        reasons.append(f"~{volume} scans/day keeps the hosted shape cheap "
                         "to consume")

    recommendation = "hosted-service" if hosted >= managed else \
        "self-managed-installation"
    reasons.append(f"score hosted={hosted} self-managed={managed}")
    assert recommendation in names
    return recommendation, reasons


ESTATES = [
    {"tier": "samples", "must_stay_inside": False, "scan_volume": 20,
     "team_runs_services": False, "audit_pressure": False},
    {"tier": "platform", "must_stay_inside": False, "scan_volume": 800,
     "team_runs_services": True, "audit_pressure": False},
    {"tier": "regulated", "must_stay_inside": True, "scan_volume": 300,
     "team_runs_services": True, "audit_pressure": True},
    {"tier": "audited-saas", "must_stay_inside": False, "scan_volume": 60,
     "team_runs_services": False, "audit_pressure": True},
]

for estate in ESTATES:
    recommendation, reasons = choose_shape(estate)
    print(f"{estate['tier']}: {recommendation}")
    for reason in reasons:
        print(f"  - {reason}")

assert choose_shape(ESTATES[0])[0] == "hosted-service"
assert choose_shape(ESTATES[1])[0] == "self-managed-installation"
assert choose_shape(ESTATES[2])[0] == "self-managed-installation"
assert choose_shape(ESTATES[3])[0] == "hosted-service"


## Verify

Every code cell asserts its own behaviour, so running the notebook top to
bottom is the verification step:

- the shape cell confirms the two shapes differ on every ownership axis;
- the residency cell confirms an unconstrained tier keeps both shapes while
  a must-stay-inside tier keeps only the self-managed one;
- the burden cell confirms self-managed costs the team more operationally
  and that the hosted shape carries no upgrade or backup burden;
- the scoring cell confirms the recommendation for four contrasting tiers,
  including the residency veto firing regardless of the other weights.

A failure means a model assumption changed, not that the notebook is wrong:
check which relationship the assertion encodes before changing it.

## Common errors

- **Choosing one shape for every tier.** The residency veto is per tier.
  Standardising on a single shape either breaks a residency constraint or
  pays self-managed operating cost where nothing required it.
- **Counting only licence or service cost.** The self-managed shape moves
  cost into upgrades, backups, availability work, and on-call time. Compare
  total burden, not sticker cost.
- **Monitoring findings but not freshness.** A scanner that stopped running
  and a codebase with no secrets look identical in the dashboard. Alert on
  missing scan results per repository, in both shapes.
- **Treating the hosted console as the policy backup.** Export and version
  the scanning policy alongside the repository configuration so a bad console
  edit can be reverted from version control.

## References

- `gitguardian/docs/cicd-secret-scanning-integration.md` — pipeline wiring
  this choice feeds into; the per-tier decision here determines which service
  each pipeline reports to.
- `gitguardian/configs/monorepo-allowlists.yaml` — allowlist pattern that
  applies in either shape; the policy content is the same, only its hosting
  differs.
- `gitguardian/templates/gitguardian-multi-repo-scanning-scaffold/` —
  multi-repository scaffold; point its scanner configuration at the chosen
  service per tier.
